In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
import os

path = "/content/drive/MyDrive/ML_aircraft_recognition/utils.py"
print("Exists:", os.path.exists(path))
print("Size:", os.path.getsize(path), "bytes")

Exists: True
Size: 4754 bytes


In [18]:
print(os.listdir("/content/drive/MyDrive/ML_aircraft_recognition"))

['FGVC-Aircraft', '03_models.ipynb', '01_EDA.ipynb', 'dataset.pkl', 'utils.py', '00_utils.ipynb', '02_training.ipynb']


In [22]:
import importlib.util
import sys

spec  = importlib.util.spec_from_file_location("utils", "/content/drive/MyDrive/ML_aircraft_recognition/utils.py")
utils = importlib.util.module_from_spec(spec)
sys.modules["utils"] = utils  # ← register it
spec.loader.exec_module(utils)

from utils import load_dataset, get_dataloaders, build_model, save_model, build_resnet, save_resnet
print("✅ Loaded from:", utils.__file__)

✅ Loaded from: /content/drive/MyDrive/ML_aircraft_recognition/utils.py


**Setup**

In [23]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append("/content/drive/MyDrive/ML_aircraft_recognition")
from utils import load_dataset, get_dataloaders, build_model, save_model, build_resnet, save_resnet

import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda


**Load data**

In [24]:
dataset = load_dataset()
train_loader, val_loader, test_loader, num_classes, train_size, val_size, test_size = get_dataloaders(dataset)

 100% |███████████████| 3334/3334 [2.1s elapsed, 0s remaining, 1.6K samples/s]       


INFO:eta.core.utils: 100% |███████████████| 3334/3334 [2.1s elapsed, 0s remaining, 1.6K samples/s]       


Loaded 3334 samples
Train: 2333 | Val: 500 | Test: 501 | Classes: 100


**Build model**

In [25]:
model = build_model(num_classes, device)
print(f"Model ready — {num_classes} classes")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 239MB/s]


Model ready — 100 classes


**Train EfficientNet**

In [26]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

EPOCHS       = 5
best_val_acc = 0.0

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    train_loss, train_correct = 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss    += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()

    # --- Validate ---
    model.eval()
    val_loss, val_correct = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs     = model(imgs)
            val_loss   += criterion(outputs, labels).item()
            val_correct += (outputs.argmax(1) == labels).sum().item()

    train_acc = train_correct / train_size
    val_acc   = val_correct   / val_size
    scheduler.step()

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss/len(val_loader):.4f}   | Val Acc: {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_model(model)
        print(f"  ✅ Best model saved (val_acc={val_acc:.4f})")

Epoch 01/5 | Train Loss: 4.5686 | Train Acc: 0.0253 | Val Loss: 4.4599   | Val Acc: 0.0700
Model saved to /content/drive/MyDrive/ML_aircraft_recognition/best_model.pth
  ✅ Best model saved (val_acc=0.0700)
Epoch 02/5 | Train Loss: 4.2026 | Train Acc: 0.1847 | Val Loss: 4.1569   | Val Acc: 0.1260
Model saved to /content/drive/MyDrive/ML_aircraft_recognition/best_model.pth
  ✅ Best model saved (val_acc=0.1260)
Epoch 03/5 | Train Loss: 3.6839 | Train Acc: 0.3416 | Val Loss: 3.6469   | Val Acc: 0.2140
Model saved to /content/drive/MyDrive/ML_aircraft_recognition/best_model.pth
  ✅ Best model saved (val_acc=0.2140)
Epoch 04/5 | Train Loss: 3.0321 | Train Acc: 0.4634 | Val Loss: 3.1047   | Val Acc: 0.2980
Model saved to /content/drive/MyDrive/ML_aircraft_recognition/best_model.pth
  ✅ Best model saved (val_acc=0.2980)
Epoch 05/5 | Train Loss: 2.4077 | Train Acc: 0.5688 | Val Loss: 2.6674   | Val Acc: 0.3580
Model saved to /content/drive/MyDrive/ML_aircraft_recognition/best_model.pth
  ✅ Best

**Train resnet**

In [29]:
resnet = build_resnet(num_classes, device)

criterion        = nn.CrossEntropyLoss()
resnet_optimizer = optim.Adam(resnet.parameters(), lr=1e-4)
resnet_scheduler = optim.lr_scheduler.StepLR(resnet_optimizer, step_size=5, gamma=0.5)

EPOCHS          = 5
best_resnet_acc = 0.0

for epoch in range(EPOCHS):
    # --- Train ---
    resnet.train()
    train_loss, train_correct = 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        resnet_optimizer.zero_grad()
        outputs = resnet(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        resnet_optimizer.step()
        train_loss    += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()

    # --- Validate ---
    resnet.eval()
    val_loss, val_correct = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs     = resnet(imgs)
            val_loss   += criterion(outputs, labels).item()
            val_correct += (outputs.argmax(1) == labels).sum().item()

    train_acc = train_correct / train_size
    val_acc   = val_correct   / val_size
    resnet_scheduler.step()

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss/len(val_loader):.4f}   | Val Acc: {val_acc:.4f}")

    if val_acc > best_resnet_acc:
        best_resnet_acc = val_acc
        save_resnet(resnet)
        print(f"  ✅ Best ResNet saved (val_acc={val_acc:.4f})")

Epoch 01/5 | Train Loss: 4.1639 | Train Acc: 0.1089 | Val Loss: 3.4766   | Val Acc: 0.1800
ResNet saved to /content/drive/MyDrive/ML_aircraft_recognition/best_resnet.pth
  ✅ Best ResNet saved (val_acc=0.1800)
Epoch 02/5 | Train Loss: 2.6476 | Train Acc: 0.4363 | Val Loss: 2.5148   | Val Acc: 0.3800
ResNet saved to /content/drive/MyDrive/ML_aircraft_recognition/best_resnet.pth
  ✅ Best ResNet saved (val_acc=0.3800)
Epoch 03/5 | Train Loss: 1.4800 | Train Acc: 0.7321 | Val Loss: 2.1502   | Val Acc: 0.4580
ResNet saved to /content/drive/MyDrive/ML_aircraft_recognition/best_resnet.pth
  ✅ Best ResNet saved (val_acc=0.4580)
Epoch 04/5 | Train Loss: 0.6983 | Train Acc: 0.9074 | Val Loss: 1.7683   | Val Acc: 0.5420
ResNet saved to /content/drive/MyDrive/ML_aircraft_recognition/best_resnet.pth
  ✅ Best ResNet saved (val_acc=0.5420)
Epoch 05/5 | Train Loss: 0.2800 | Train Acc: 0.9799 | Val Loss: 1.7922   | Val Acc: 0.5480
ResNet saved to /content/drive/MyDrive/ML_aircraft_recognition/best_resne